## Homework 1: VQGAN Transformer

Мы привыкли, что для работы с изображениями нужны сверточные сети. Но картинку можно превратить в последовательность токенов и скормить классическому GPT точно так же, как текст.

Первыми эту идею стали использовать в **[ImageGPT](https://cdn.openai.com/papers/Generative_Pretraining_from_Pixels_V2.pdf)**. Цвета пикселей кластеризовали алгоритмом K-Means и получили некоторый словарь оттенков. Дальше авторы просто обучали обычный GPT предсказывать цвет следующего пикселя. Но такой подход быстро уперся в проблему с вычислениями, поскольку даже небольшой размер картинки $32\times32$ требуют генерации больше тысячи токенов.

Авторы **[VQGAN](https://arxiv.org/abs/2012.09841)** нашли более изящное решение. Вместо того чтобы квантовать пиксели напрямую, они сначала обучают VQ-VAE, который сжимает картинку в компактную сетку токенов, например $16\times16$. Уже поверх этой короткой последовательности учится тот же авторегрессионный трансформер, только теперь каждый токен несёт на порядки больше смысла.

### Задание
В этой работе мы попробуем реализовать рецепт из статьи VQGAN и обучить GPT для генерации картинок. Обучать свой VQ-VAE нам не придется, поскольку это потребует значительных ресурсов. Вместо этого мы будем использовать предобученную модель, с помощью которой закодируем наш датасет, и обучим поверх полученных токенов трансформер.

За выполнение домашнего задания можно получить до **10 баллов**.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import VQModel
from IPython.display import clear_output
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

### Задание 1: Dataset (0.5 балла)

Для обучения нашей модели мы будем использовать датасет `Imagenette` — уменьшенную версию ImageNet из 10 классов c  $\sim 950$ фотографиями на класс.

Датасет уже разбит на `train` и `val`. Мы возьмём вариант с `size="160px"`, этого с запасом хватит для наших целей.

In [ ]:
from torchvision.datasets import Imagenette

Картинки тут не фиксированного размера, поэтому перед подачей в модель их нужно обрезать:

- Создайте преобразования `transforms` для обучающего (`train_dataset`) — `Resize(144)`, `RandomCrop(128)` и `RandomHorizontalFlip`.
- Для валидационного (`val_dataset`) — `Resize(144)` и `CenterCrop(128)`.
- Загрузите Imagenette (`size="160px"`) с соответствующими преобразованиями
- Создайте DataLoaders для каждого датасета

In [ ]:
# Some images are grayscale (mode "L"), not RGB - convert them explicitly.
to_rgb = transforms.Lambda(lambda img: img.convert("RGB"))

# Train transform: Resize(144), RandomCrop(128), RandomHorizontalFlip, ToTensor
train_transform = ...

# Val transform: same Resize(144) + CenterCrop(128), but no augmentations
val_transform = ...

train_dataset = ...
val_dataset = ...

# Human-readable class names - train_dataset.classes is a list of synonym tuples
mappings = {i: names[0] for i, names in enumerate(train_dataset.classes)}

batch_size = ...
num_workers = ...
to_use_pin_memory = ...

train_loader = ...
val_loader = ...

Прежде чем приступить к построению модели всегда полезно взглянуть на данные, с которыми предстоит работать. Это даёт общее представление о данных и позволяет убедиться, что всё загрузилось корректно.

In [ ]:
# Select a random set of indices from the training dataset.
indices = torch.randperm(len(train_dataset))[:40]

fig, axes = plt.subplots(5, 8, figsize=(16, 10))

for i, ax in enumerate(axes.flat):
    image, label = train_dataset[indices[i]]  

    image = image.permute(1, 2, 0).cpu().numpy()

    ax.imshow(image)
    ax.axis('off')
    ax.set_title(f"{mappings[label]}", fontsize=8)

plt.tight_layout()
plt.show()

### Задание 2: Предобученный VQ-VAE (1.5 балла)

Чтобы получить латентную сетку токенов из картинки, мы будем использовать VQ-VAE, который состоит из 3 частей:

- **Encoder** сжимает картинку в сетку непрерывных векторов.
- **Vector Quantizer** заменяет каждый вектор на ближайший вектор из обучаемого словаря (codebook).
- **Decoder** восстанавливает картинку обратно по этой квантованной сетке.

<center><img src="images/vq-vae.png" width=700></center>

Обучить хороший VQ-VAE с нуля непросто. Поэтому мы возьмём предобученный VQ-VAE из библиотеки `diffusers` (`microsoft/vq-diffusion-ithq`) и будем использовать его только для кодирования и декодирования картинок.

На что стоит обратить внимание:

- модель уменьшает разрешение в 8 раз
- размер codebook (`num_vq_embeddings`) — 4096 векторов
- вход нормализован в $[-1, 1]$, а не в $[0, 1]$, как отдаёт `ToTensor()`

Для наших картинок $128\times128$ это даёт латентную сетку $16\times16$, то есть $256$ токенов.

In [ ]:
vqvae = VQModel.from_pretrained("microsoft/vq-diffusion-ithq", subfolder="vqvae")
vqvae.to(device)
vqvae.eval()

for p in vqvae.parameters():
    p.requires_grad = False

print(f"Codebook size: {vqvae.config.num_vq_embeddings}")

Дальше нам нужно научиться переводить картинку в латентные токены и обратно. Для этого реализуйте 3 функции:

- **`img_to_codes`**: кодирует картинку в сетку индексов нашего словаря

- **`codes_to_img`**: восстанавливает картинку по сетке индексов

- **`img_to_seq`**: преобразует двумерную сетку латентных токенов $16\times16$ в одномерную последовательность.

In [ ]:
def img_to_codes(x, vqvae):
    # x: [B, 3, H, W] in [0, 1] range (as returned by ToTensor())

    # Rescale x to [-1, 1], the range VQModel is trained on
    x = ...

    # Get the continuous encoder latents
    latent = ...

    # Quantize the latents and get the indices of the nearest codebook vectors
    indices = ...

    b, _, h, w = latent.shape
    return indices.view(b, h, w)


def codes_to_img(codes, vqvae):
    # codes: [B, H, W] long - indices into the codebook
    b, h, w = codes.shape
    flat_codes = codes.reshape(-1)

    # Look up codebook vectors by index (use vqvae.quantize.get_codebook_entry)
    quant = ...

    # Run through vqvae.decode
    dec = ...

    return ...  # back to [0, 1] for display


def img_to_seq(x):
    # x: [batch_size, H, W] -> [batch_size, H * W]
    # Flatten the token grid into a row
    b, h, w = x.shape

    return ...

In [ ]:
def plot_reconstructions(vqvae, val_loader, device, n=8):
    images, _ = next(iter(val_loader))
    images = images[:n].to(device)
    with torch.no_grad():
        codes = img_to_codes(images, vqvae)
        recon = codes_to_img(codes, vqvae)

    originals = images.permute(0, 2, 3, 1).cpu().numpy()
    recons = recon.permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy()

    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        axes[0, i].imshow(originals[i])
        axes[1, i].imshow(recons[i])

    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    axes[0, 0].set_ylabel("original")
    axes[1, 0].set_ylabel("recon")
    plt.tight_layout()
    plt.show()

plot_reconstructions(vqvae, val_loader, device)

### Задание 3: Embeddings (1.5 балла)

Чтобы трансформер мог работать с изображениями, их нужно преобразовать в понятный для него формат. В текстовых моделях слова сначала превращаются в  токены, а затем — в вектора эмбеддингов, в которых хранится "смысл слов".

С изображениями мы поступаем так же, но вместо слов мы работаем с токенами VQ-VAE. Их превращают в векторы, которые в процессе обучения постепенно учатся отражать некоторые связи между токенами помогают модели улавливать контекст изображения.

В ImageGPT нам потребуются несколько типов эмбеддингов для кодирования входных изображений:

- **`token_embeddings`**: преобразуют индексы латентных токенов в вектора

- **`position_embeddings`**: кодируют информацию о позиции каждого токена в последовательности, это позволяет модели улавливать пространственные отношения между ними

- **`class_embedding`**: преобразует индекс класса в вектор и выступает в роли условного стартового токена, который сообщает модели, изображение какого именно класса нужно генерировать

In [ ]:
class GPTEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_positions, num_classes):
        super().__init__()
        self.token_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.position_embeddings = nn.Embedding(max_positions, embed_dim)
        self.class_embedding = nn.Embedding(num_classes, embed_dim)

    def forward(self, x, cls_label):
        # x: [batch_size, seq_len], cls_label: [batch_size]

        tok_emb = ...  
        cls_token_emb = ...

        # Prepend the class embedding to the start of the sequence
        full_seq = ...  # [batch_size, seq_len + 1, embed_dim]

        # Positional embeddings for the resulting length, with a batch axis for broadcasting
        pos_ids = ...  # [seq_len + 1]
        pos_emb = ...  # [1, seq_len + 1, embed_dim]

        embeddings = ...  # [batch_size, seq_len + 1, embed_dim]

        return embeddings

### Задание 4: Decoder Block (1.5 балла)

Теперь нам нужно собрать основные блоки нашей модели — блоки декодера. В каждом таком блоке последовательно выполняются следующие операции:

- **`LayerNorm`**
- **`MultiHeadAttention`**
- **`Residual Connection`**
- **`LayerNorm`**
- **`MLP`**:
  - **Linear**: $embed\; dim$ -> $4 \times embed\; dim$
  - **GELU Activation**
  - **Linear**: $(4 \times embed\; dim)$ -> $embed\; dim$
- **`Residual Connection`**

Поскольку наша модель является генеративной и предсказывает следующий элемент в последовательности, вам потребуется использовать **Masked Self-Attention**. Сама маска зависит только от длины последовательности, поэтому блок не строит её сам, а получает готовой аргументом `attn_mask` (её создаст `ImageGPT` в Задании 5).

Вы можете воспользоваться готовой реализацией **Multi-Head Attention** из PyTorch.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.ln1 = ...
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            batch_first=True,  # our batch format is [batch_size, seq_len, embed_dim]
        )
        self.ln2 = ...
        self.mlp = ...

    def forward(self, x, attn_mask):
        # x: [batch_size, seq_len + 1, embed_dim]
        # attn_mask: [seq_len + 1, seq_len + 1] - ready-made causal mask, built by ImageGPT

        # 1. LayerNorm + Masked Multi-Head Attention + Residual
        residual = x
        x = ...

        attn_output, _ = ...
        x = ...

        # 2. LayerNorm + MLP + Residual
        residual = x
        x = ...
        mlp_output = ...
        x = ...

        return x

### Задание 5: Final Model (1 балл)

Теперь, когда у нас есть все компоненты, мы можем собрать их вместе, чтобы построить итоговую модель ImageGPT.

Итоговая архитектура выглядит так:

- **`Embedding`**: преобразует входные токены в векторы

- **`Blocks`**: использует `num_layers` блоков `DecoderBlock`, которые последовательно обрабатывают наши данные

- **`Final LayerNorm`**: нормализует выходные данные перед финальным предсказанием

- **`Head`**: преобразует обработанные векторы в предсказания

Здесь же создайте causal-маску на максимальную длину последовательности и положите её в буфер модели. В `forward` от неё достаточно отрезать кусок под текущую длину и передать во все блоки, так она построится один раз, а не заново в каждом блоке на каждом форварде.

Обратите внимание, что `ImageGPT` работает только с индексами токенов в словаре и ничего не знает о цвете — за декодирование обратно в картинку отвечает загруженный ранее `vqvae` (см. Задание 2).

In [ ]:
class ImageGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, max_positions, num_classes):
        super().__init__()
        self.embedding = ...
        self.blocks = ...  # nn.ModuleList of num_layers DecoderBlock
        self.ln_f = ...
        self.head = ...  # nn.Linear without bias, embed_dim -> vocab_size

        # Causal mask built once for the max sequence length and stored as a buffer
        causal_mask = ...
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x, y):
        # x: [batch_size, seq_len], y: [batch_size]

        h = ...  # [batch_size, seq_len + 1, embed_dim]

        # Slice the mask down to the current sequence length
        seq_len = h.size(1)
        attn_mask = ...

        for block in self.blocks:
            h = ...

        h = ...  # final LayerNorm
        output = ...  # project to vocabulary logits

        return output

In [ ]:
model = ImageGPT(
    embed_dim=512,
    num_heads=8,
    num_layers=24,
    max_positions=16 * 16,
    vocab_size=vqvae.config.num_vq_embeddings,
    num_classes=10
    )

total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

### Задание 6: Sampling (1.5 балла)

Для генерации новых изображений мы будем использовать уже обученную модель ImageGPT. Давайте напишем функцию **`sample`**, которая будет отвечать за это.

Она будет принимать на вход следующие параметры:

- **`model`**: наша обученная модель ImageGPT.

- **`length`**: число шагов генерации

- **`context`**: начальная последовательность латентных токенов, если мы хотим дорисовать изображение. Если её нет, генерация начнётся с нуля.

- **`class_label`**: метка класса для генерации по заданному условию

- **`temperature`**: параметр, который регулирует уровень случайности при генерации

- **`num_samples`**: количество изображений, которые нужно сгенерировать

- **`top_k`**: количество самых вероятных токенов, которые будут использоваться при генерации

- **`device`**: устройство, на котором будет происходить генерация, по умолчанию берётся то, на котором лежит модель

In [ ]:
def sample(model, length, class_label, num_samples=1, device=None,
           context=None, temperature=1.0, top_k=None):

    if device is None:
        device = next(model.parameters()).device

    # Expand the class label so every generated image gets its own
    class_label = ...  # [num_samples]

    if context is None or context.numel() == 0:
        context = ...  # [num_samples, 0]
    else:
        if context.dim() == 1:
            context = ...  # add a batch axis: [1, context_len]
        context = ...  # repeat the context num_samples times: [num_samples, context_len]

    generated = context  # [num_samples, context_len]

    with torch.no_grad():
        for _ in range(length):
            logits = ...  # [num_samples, context_len + 1, vocab_size]

            # Logits for the next token (last position of the model output)
            next_token_logits = ...  # [num_samples, vocab_size]

            if top_k is not None:
                top_values, _ = ...
                threshold = ...
                next_token_logits = ...

            probs = ... 
            next_tokens = ...

            generated = ...

    return generated

### Задание 7: Training and Validation Loop (1.5 балла)

После того как все компоненты модели собраны, мы можем приступить к обучению. Вам нужно реализовать три функции:

- **`compute_loss`**: кодирует батч картинок в токены и считает по ним ошибку.

- **`train_one_epoch`**: проходит одну эпоху по обучающей выборке

- **`validate_one_epoch`**: роходит одну эпоху по валидационной выборке

Функции **`save_checkpoint`** и **`plot_losses`** уже написаны, они сохраняют веса и рисуют график лоссов по эпохам.

In [ ]:
def compute_loss(model, vqvae, images, labels, criterion):
    with torch.no_grad():
        encoding_indices = ...  # [batch_size, 16, 16]
        tokens = ...  # [batch_size, 256]

    logits = ...

    return ...


def train_one_epoch(model, vqvae, train_loader, optimizer, criterion, scheduler, device):

    model.train()
    total_loss = 0.0

    for images, labels in tqdm(train_loader, desc="Train"):
        images = images.to(device)
        labels = labels.to(device)

        loss = ...
        ...  # backward, optimizer step, zero the gradients, and step the scheduler (if any)

        total_loss += ...

    return total_loss / len(train_loader)


def validate_one_epoch(model, vqvae, val_loader, criterion, device):

    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Val"):
            images = images.to(device)
            labels = labels.to(device)

            total_loss += ...

    return total_loss / len(val_loader)

def save_checkpoint(model, save_path):
    save_dir = os.path.dirname(save_path)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)

    torch.save(model.state_dict(), save_path)
    print(f"Checkpoint saved: {save_path}")

def plot_losses(train_losses, val_losses):
    clear_output()
    plt.figure(figsize=(6, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

Для визуализации результатов работы нашей модели нам потребуется несколько вспомогательных функций. Ниже представлены некоторые функции, которые помогут нам подготовить данные, запустить процесс генерации и сравнить сгенерированные изображения с оригиналом.

In [ ]:
def pick_val_sample(val_loader, device):
    img_sample, class_sample = next(iter(val_loader))
    img_sample = img_sample[0:1].to(device)
    class_sample = class_sample[0:1].to(device)
    return img_sample, class_sample

def generate_image_tokens(model, class_sample, h, w, device, top_k=None):
    gen_seq = sample(
        model,
        context=None,
        class_label=class_sample,
        length=h * w,
        device=device,
        top_k=top_k
    )  # [1, h * w]
    return gen_seq.squeeze(0)  # [h * w]

def image_to_tokens(image_tensor, vqvae):
    with torch.no_grad():
        encoding_indices = img_to_codes(image_tensor, vqvae)  # [1, h, w]
    return encoding_indices.reshape(-1)  # [h * w]

def tokens_to_image(tokens, h, w, vqvae):
    tokens = tokens.reshape(1, h, w).to(next(vqvae.parameters()).device)
    with torch.no_grad():
        rgb_image = codes_to_img(tokens, vqvae).clamp(0, 1)  # [1, 3, 128, 128]

    rgb_image = (rgb_image[0] * 255).to(torch.uint8)  # [3, 128, 128]

    return rgb_image.permute(1, 2, 0).cpu().numpy()  # [128, 128, 3]

def plot_generated_vs_original(gen_img_rgb, original_img_rgb, class_label):
    final_img = np.concatenate([gen_img_rgb, original_img_rgb], axis=1)
    plt.figure(figsize=(4, 2))
    plt.title(f"Generated vs Original – Class {mappings[class_label]}")
    plt.imshow(final_img)
    plt.axis("off")
    plt.show()

def generate_and_plot_sample(model, vqvae, val_loader, device, top_k=None):
    model.eval()
    img_sample, class_sample = pick_val_sample(val_loader, device)

    with torch.no_grad():
        codes = img_to_codes(img_sample, vqvae)  # [1, h, w]
    h, w = codes.shape[1], codes.shape[2]
    original_tokens = codes.reshape(-1)  # [h * w]

    gen_seq = generate_image_tokens(model, class_sample, h, w, device, top_k)
    generated_image_rgb = tokens_to_image(gen_seq, h, w, vqvae)
    original_image_rgb = tokens_to_image(original_tokens, h, w, vqvae)
    
    plot_generated_vs_original(generated_image_rgb, original_image_rgb, class_sample.item())

In [ ]:
def train_imagegpt(model, vqvae, train_loader, val_loader, optimizer, criterion,
                   scheduler, checkpoint_dir, device=device, epochs=10):

    model.to(device)
    vqvae.to(device)
    vqvae.eval()

    train_loss_history, val_loss_history = [], []
    best_val_loss = float("inf")

    for epoch in range(epochs):

        train_loss = ...
        train_loss_history.append(train_loss)
        val_loss = ...
        val_loss_history.append(val_loss)

        plot_losses(train_loss_history, val_loss_history)
        print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

        save_checkpoint(model, f"{checkpoint_dir}/imagegpt_last.pt")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(model, f"{checkpoint_dir}/imagegpt_best.pt")

        # you can comment this out if you don't want to generate samples after every epoch
        generate_and_plot_sample(model, vqvae, val_loader, device)

Теперь мы готовы запустить основной цикл обучения. Вы можете экспериментировать с различными гиперпараметрами и расписанием обучения, чтобы добиться хороших результатов.

In [ ]:
epochs = 50
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
checkpoint_dir = "checkpoints_imagegpt"

train_imagegpt(
    model=model,
    vqvae=vqvae,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    checkpoint_dir=checkpoint_dir,
    device=device,
    epochs=epochs
)

После обучения загрузим лучшие по валидации веса, чтобы дальше использовать их для генерации.

In [ ]:
checkpoint_file = "checkpoints_imagegpt/imagegpt_best.pt"

model.to(device)
model.load_state_dict(torch.load(checkpoint_file, map_location=device))
model.eval()
print(f"Model loaded from: {checkpoint_file}")

### Задание 8: Autocompletion (1 балл)

Посмотрим, как модель дорисовывает картинку по началу последовательности токенов. Мы будем давать ей первые `context_size` токенов настоящего изображения и просить сгенерировать все остальные.

Функции для генерации и отрисовки уже написаны, разбираться в их коде необязательно. Главная из них — `generate_and_plot_variations`, именно её нужно запускать с разными параметрами в заданиях 8.1–8.3 ниже.

Каждая строка результата — это один пример из валидации:

- **`context`** (слева): та часть картинки, которую модель получила на вход. Невидимая ей часть закрашена серым.
- **`sample 1..N`**: разные варианты продолжения этого же контекста, отличаются только случайностью сэмплирования.
- **`original`** (справа): как эта картинка выглядит на самом деле.

Обратите внимание, что `original` — это не исходное фото, а его реконструкция через VQ-VAE. Сравнивать генерации честно именно с ней, потому что лучше своего токенизатора модель нарисовать не может.

In [ ]:
def pick_val_contexts(val_loader, num_contexts, device, seed=0):
    dataset = val_loader.dataset
    generator = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(dataset), generator=generator)

    context_images, class_labels = [], []
    seen_classes = set()
    with torch.no_grad():
        for idx in perm:
            img, label = dataset[idx.item()]
            if label in seen_classes:
                continue
            seen_classes.add(label)
            context_images.append(img.to(device))
            class_labels.append(torch.tensor(label, device=device))
            if len(context_images) >= num_contexts:
                break
    return context_images, class_labels

def prepare_context(img_tensor, context_size, vqvae):
    original_tokens = image_to_tokens(img_tensor.unsqueeze(0), vqvae)  # [h * w]
    context_tensor = original_tokens[:context_size]
    return context_tensor, original_tokens

def generate_variations(model, vqvae, context_tensor, class_label, h, w, context_size, samples_per_context, temperature=1.0, top_k=None):
    total_tokens = h * w

    gen_seq = sample(model,
                     length=total_tokens - context_size,
                     context=context_tensor,
                     class_label=class_label,
                     num_samples=samples_per_context,
                     temperature=temperature,
                     device=context_tensor.device,
                     top_k=top_k)

    return [tokens_to_image(gen_seq[i], h, w, vqvae) for i in range(samples_per_context)]

def make_context_preview(original_img_rgb, h, w, context_size):
    seen = np.zeros(h * w, dtype=bool)
    seen[:context_size] = True

    scale = original_img_rgb.shape[0] // h
    seen_px = np.repeat(np.repeat(seen.reshape(h, w), scale, axis=0), scale, axis=1)

    preview = original_img_rgb.copy()
    preview[~seen_px] = 128

    return preview

def plot_context_rows(rows_of_images, class_labels):
    num_contexts = len(rows_of_images)
    if num_contexts == 0:
        return
    num_cols = len(rows_of_images[0])
    col_labels = ["context"] + [f"sample {j + 1}" for j in range(num_cols - 2)] + ["original"]

    fig, axs = plt.subplots(nrows=num_contexts, ncols=1, figsize=(num_cols * 2, num_contexts * 2.4))
    if num_contexts == 1: 
        axs = [axs]

    for i, (row, label) in enumerate(zip(rows_of_images, class_labels)):
        combined_row_img = np.concatenate(row, axis=1)
        axs[i].imshow(combined_row_img)
        axs[i].set_title(f"Class: {mappings[label]}")

        tile_w = combined_row_img.shape[1] // num_cols
        axs[i].set_xticks([tile_w * (j + 0.5) for j in range(num_cols)])
        axs[i].set_xticklabels(col_labels, fontsize=8)
        axs[i].tick_params(axis="x", length=0)
        axs[i].set_yticks([])
        for spine in axs[i].spines.values():
            spine.set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def generate_and_plot_variations(model, vqvae, val_loader, num_contexts=5, samples_per_context=5, context_size=128, temperature=1.0, top_k=None, device=None):
    
    model.eval()
    vqvae.eval()

    if device is None:
        device = next(model.parameters()).device
    
    context_images, class_labels = pick_val_contexts(val_loader, num_contexts, device)

    with torch.no_grad():
        h, w = img_to_codes(context_images[0].unsqueeze(0), vqvae).shape[1:]

    all_rows_for_plotting = []
    plot_labels = [label.item() for label in class_labels]

    for img, label in tqdm(zip(context_images, class_labels), total=num_contexts):
        context_tensor, original_tokens = prepare_context(img, context_size, vqvae)
        generated_imgs_rgb = generate_variations(
            model, vqvae, context_tensor, label.unsqueeze(0), h, w,
            context_size, samples_per_context, temperature=temperature, top_k=top_k
        )
        original_img_rgb = tokens_to_image(original_tokens, h, w, vqvae)
        context_img_rgb = make_context_preview(original_img_rgb, h, w, context_size)

        row = [context_img_rgb] + generated_imgs_rgb + [original_img_rgb]
        all_rows_for_plotting.append(row)

    plot_context_rows(all_rows_for_plotting, plot_labels)

#### Задание 8.1: Размер контекста

Запустите функцию `generate_and_plot_variations` три раза с разными значениями `context_size = {64, 128, 192}` — это $1/4$, $1/2$ и $3/4$ от полной последовательности из $256$ токенов. Остальные параметры установите равными `temperature=1.0` и `top_k=1000` во всех трёх запусках.

Обратите внимание, как меняется качество сгенерированных изображений в зависимости от того, сколько контекста получила модель, и сделайте выводы.

In [ ]:
#╰( ͡° ͜ʖ ͡°)つ──☆*:・ﾟ 
# Your code here

In [ ]:
#╰( ͡° ͜ʖ ͡°)つ──☆*:・ﾟ 
# Your code here

In [ ]:
#╰( ͡° ͜ʖ ͡°)つ──☆*:・ﾟ 
# Your code here

**Ваш ответ:**

#### Задание 8.2: Температура

Теперь исследуем влияние температуры на качество и разнообразие изображений.

Запустите функцию `generate_and_plot_variations` три раза, используя одинаковый размер контекста `context_size=128` и `top_k=1000`, но с разными значениями температуры `temperature = {0.9, 0.7, 0.5}`. Посмотрите на результаты и сделайте выводы.

In [ ]:
#╰( ͡° ͜ʖ ͡°)つ──☆*:・ﾟ 
# Your code here

In [ ]:
#╰( ͡° ͜ʖ ͡°)つ──☆*:・ﾟ 
# Your code here

In [ ]:
#╰( ͡° ͜ʖ ͡°)つ──☆*:・ﾟ 
# Your code here

**Ваш ответ:**

#### Задание 8.3: Top-k

Наконец, посмотрим на `top_k`.

Запустите функцию `generate_and_plot_variations` три раза с `context_size=128` и `temperature=1.0`, но с разными значениями `top_k = {50, 500, None}`. Значение `None` означает, что отсечения нет и модель выбирает из всех $4096$ токенов словаря.

Сравните, как `top_k` влияет на связность картинки и на разнообразие вариантов, и подумайте, чем этот эффект отличается от того, что даёт температура.

In [ ]:
#╰( ͡° ͜ʖ ͡°)つ──☆*:・ﾟ 
# Your code here

In [ ]:
#╰( ͡° ͜ʖ ͡°)つ──☆*:・ﾟ 
# Your code here

In [ ]:
#╰( ͡° ͜ʖ ͡°)つ──☆*:・ﾟ 
# Your code here

**Ваш ответ:**

In [ ]:
# Здесь можно оставить отзывы, пожелания и впечатления о ДЗ :)